**BERN02 Exercise: Hierarchical Models and Testing**

Name: Yang Shann Wen

Date: 13 September 2026

In this exercise, we analyse a data set of a collection of 7 case-control studies examining the effectiveness of descriptive social norms on hotel customers' behavior to reuse their towels.

* **Response variable (y):** Number of customers reusing hotel towel (`reuse`)
* **Trial size (n):** Total number of customers observed (`total`)
* **Predictor (x)**: Experimental condition (`group`), where x=0 corresponds to the control group, and x=1 corresponds to the social norm group

** **
**1. Formulating regression model**

We formulate a binomial regression model, because the response variable represents counts of successes (customers who reused their towel) out of a fixed number of independent trials (total customers observed).

$$y \sim \text{Binomial}(n, p)$$
* **y**: Number of customers reusing hotel towel (`reuse`)

* **n**: Total number of customers observed (`total`)

* **p**: Probability of customer reusing their towel

****
**2. Hierarchical structure**

The regression model is a hierarchical model, as the data comes from 7 different studies conducted in different hotel settings. The baseline probability of towel reuse may vary between studies, so between-study heterogeneity should be taken into account.

A linear model with random and fixed effects is formulated:  
$$\text{logit}(p_{ij}) = \beta_0 + u_j + \beta_1 x_{ij}$$

* **$p_{ij}$:** Probability of towel reuse for group i in study j
* **$\beta_0$:** Overall baseline log-odds of towel reuse in the control group (intercept)
* **$u_j \sim\ Normal(0,τ^2)$:** Random intercept for study j, representing baseline differences across the 7 studies
* **$\beta_1$:** Fixed effect of the intervention
* **$x_{ij}$:** Indicator variable for the study group

---
**3. Parameter Estimating using Bayesian Inference**

We estimate the parameters of our hierarchical model, specifically the intervention effect $\beta_1$, using Bayesian inference via the `bambi` package.

Using the posterior distribution, we will summarize the intervention effect $\beta_1$ using the posterior mean and 90% probability interval (HDI).

In [38]:
import pandas as pd
import numpy as np
import arviz as az
import bambi as bmb
import warnings

In [39]:
# Import data
data_file = "towelData.csv"
data = pd.read_csv(data_file, sep=';', encoding='latin1')
count = data.iloc[:, -1] # get the last column with numbers

# Data wrangling
# count has the number of yes and no for control and social norm groups
control_yes = count[::4].to_numpy() # every 4th starting from 0 - control group + yes
control_no = count[2::4].to_numpy() # every 4th starting from 2 - control group + no
control_total = np.array([y + n for y, n in zip(control_yes, control_no)])

social_yes = count[1::4].to_numpy() # every 4th starting from 1 - social norm group + yes
social_no = count[3::4].to_numpy() # every 4th starting from 3 - social norm group + no
social_total = np.array([y + n for y, n in zip(social_yes, social_no)])

study = np.arange(1,len(control_yes)+1) # 7 different studies

control_data = pd.DataFrame({"reuse": control_yes, "total": control_total, "group": "control", "study": study})
social_data = pd.DataFrame({ "reuse": social_yes, "total": social_total, "group": "social", "study": study})

combined_data = pd.concat([control_data, social_data], ignore_index=True)

# Convert data types - important for bambi that they are correctly set
# can give errors otherwise
combined_data['reuse'] = combined_data['reuse'].astype(int) # Number of reuses
combined_data['total'] = combined_data['total'].astype(int) # Total guests
combined_data['group'] = combined_data['group'].astype('category') # Group (control / social)
combined_data['study'] = combined_data['study'].astype('category') # Study number

In [40]:
# Look into the combined_data
print(combined_data)

    reuse  total    group study
0      74    211  control     1
1     103    277  control     2
2      77    135  control     3
3      82    187  control     4
4      21     25  control     5
5     123    147  control     6
6      28     30  control     7
7      98    222   social     1
8     587   1318   social     2
9     406    655   social     3
10    278    555   social     4
11     21     24   social     5
12    472    576   social     6
13    101    132   social     7


In [41]:
# For reproducibility
random_seed = 1234

In [42]:
# Formulating the hierarchical binomial model
# p(reuse, total): models the probability of reusing towels out of the total observed customers
# ~ group: fixed effect estimating the overall impact of the social norm message vs. control
# (1|study): random intercept giving each of the 7 studies its own baseline reuse rate to account for study-to-study differences

model = bmb.Model("p(reuse, total) ~ group + (1|study)", combined_data, family="binomial")
model

       Formula: p(reuse, total) ~ group + (1|study)
        Family: binomial
          Link: p = logit
  Observations: 14
        Priors: 
    target = p
        Common-level effects
            Intercept ~ Normal(mu: 0.0, sigma: 1.5)
            group ~ Normal(mu: 0.0, sigma: 1.0)
        
        Group-level effects
            1|study ~ Normal(mu: 0.0, sigma: HalfNormal(sigma: 2.5495))

In [43]:
# Fitting the model and estimating the parameter
idata_hierarchical = model.fit(
    random_seed=random_seed,
    target_accept=0.90
)

Output()

In [48]:
# Have a look at the summary of the posterior distribution with a 90% probability interval
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    summary = az.summary(idata_hierarchical, hdi_prob=0.90)

summary

Array contains NaN-value.


,mean,sd,hdi_5%,hdi_95%,mcse_mean,mcse_sd,ess_bulk,ess_tail,r_hat
"(observed_data, p(reuse, total))",NaN,NaN,21.000,NaN,NaN,NaN,NaN,NaN,NaN
"('posterior', '1|study')[0]",-0.902,0.449,-1.628,-0.214,0.026,0.027,319.0,419.0,1.00
"('posterior', '1|study')[1]",-0.831,0.443,-1.545,-0.150,0.026,0.028,320.0,394.0,1.00
"('posterior', '1|study')[2]",-0.107,0.446,-0.835,0.592,0.026,0.029,320.0,320.0,1.00
"('posterior', '1|study')[3]",-0.601,0.445,-1.342,0.052,0.026,0.028,332.0,360.0,1.00
"('posterior', '1|study')[4]",1.160,0.557,0.300,2.066,0.025,0.026,541.0,546.0,1.00
"('posterior', '1|study')[5]",0.977,0.450,0.183,1.612,0.026,0.029,334.0,282.0,1.00
"('posterior', '1|study')[6]",0.787,0.470,0.042,1.543,0.025,0.026,383.0,422.0,1.00
"(posterior, 1|study_sigma)",1.105,0.392,0.534,1.662,0.021,0.018,360.0,470.0,1.00
"(posterior, Intercept)",0.388,0.446,-0.226,1.183,0.026,0.028,323.0,382.0,1.00


The posterior estimates of the model parameters are summarized below:

| Parameter | Posterior mean | 90% HDI |
|---|---:|---:|
| $\beta_0$ (Intercept) | 0.388 | [-0.226, 1.183] |
| $\beta_1$ (Group) | 0.209	| [0.084, 0.341] |
| $\tau$ (Study-level SD) | 1.105 | [0.534, 1.662 ] |

Looking at the posterior summary row for group ($\beta_1$), the estimated posterior mean is 0.210.The 90% Highest Density Interval (HDI), given by the hdi_5% and hdi_95% columns, ranges from **[0.084, 0.341]**.

Because zero is not included in this interval, the data provide strong evidence that the descriptive social norm intervention has a positive effect, increasing the odds of customers reusing their hotel towels.



**4. Formulating hypothesis**

After estimating the parameter, we need to formulate our hypothesis to test the effectiveness of the intervention.

* **Null Hypothesis ($H_0$)**: $\beta_{1}\leq 0$

  The descriptive social norm intervention has no effect, or reduces towel reuse relative to the control group.

* **Alternative Hypothesis ($H_1$)**: $\beta_{1}\ > 0$

  The descriptive social norm intervention increases towel reuse relative to the control group.

----

**5. Test the hypothesis using Bayesian Inference**

The posterior samples of the intervention effect ($\beta_{1}$), were extracted from the fitted model. The Bayesian p-value is calculated as the proportion of posterior samples for which $\beta_1\leq 0$. This represents the posterior probability that the intervention has no positive effect on towel reuse.




In [45]:
# Extract posterior samples of the intervention effect
group_samples = idata_hierarchical.posterior["group"].values.flatten()

# Calculate the Bayesian p-value
p_value = np.mean(group_samples <= 0)

print("Bayesian p-value:", p_value)

Bayesian p-value: 0.003


As a result, the Bayesian p-value is 0.003, meaning that only 0.3% of the posterior samples have $(\beta_1\leq0)$. This provides strong evidence against the null hypothesis and shows a positive intervention effect. Therefore, we conclude that the descriptive social norm intervention has a positive effect on towel reuse.